In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
import statsmodels.graphics.tsaplots
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import statistics

In [3]:
data = pd.read_csv('Data/preprocessed/NP15_rt_series.csv')
data['begin_time'] = pd.to_datetime(data['begin_time'])
data_15min = data[data['begin_time'].apply(lambda x : x.minute%15 == 0)].copy()
data_15min

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price
0,2021-01-01 00:00:00,30.76070,NaN,NaN
3,2021-01-01 00:15:00,31.35044,NaN,NaN
6,2021-01-01 00:30:00,32.45147,NaN,NaN
9,2021-01-01 00:45:00,29.98897,NaN,NaN
12,2021-01-01 01:00:00,29.89887,NaN,NaN
...,...,...,...,...
420753,2024-12-31 22:45:00,45.92347,6.20218,39.72129
420756,2024-12-31 23:00:00,46.50217,8.26084,38.24133
420759,2024-12-31 23:15:00,46.25834,7.86632,38.39202
420762,2024-12-31 23:30:00,45.51538,9.08338,36.43200


In [4]:
# Get data between specified dates
def filter_times(df, time1, time2):
    return df.apply(lambda x: (time1 <= x['begin_time']) and (x['begin_time'] <= time2) , axis=1)

In [5]:
#data = data[filter_times(data, datetime.datetime(2021, 3, 12, 0, 0, 0), datetime.datetime(2026, 1, 1, 0, 0, 0))]
#data = data.reset_index(drop=True)

In [6]:
data_train=data_15min.iloc[:112204]
data_train

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price
0,2021-01-01 00:00:00,30.76070,NaN,NaN
3,2021-01-01 00:15:00,31.35044,NaN,NaN
6,2021-01-01 00:30:00,32.45147,NaN,NaN
9,2021-01-01 00:45:00,29.98897,NaN,NaN
12,2021-01-01 01:00:00,29.89887,NaN,NaN
...,...,...,...,...
336597,2024-03-14 17:45:00,30.46712,4.35904,26.10808
336600,2024-03-14 18:00:00,30.09395,9.17368,20.92027
336603,2024-03-14 18:15:00,30.30745,-1.41394,31.72139
336606,2024-03-14 18:30:00,41.39781,4.69035,36.70746


In [8]:
def crossValidateFinal(year, n_splits, test_size, seas_order, non_seas_order):
    ts_split = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    mse_list = []
    mse_base_list = []

    print(f"Seasonal order - {seas_order}, Non-seasonal order - {non_seas_order}")

    for j in range(3,13, 3):
        time1 = datetime.datetime(year, j, 1, 0, 0, 0)
        time2 = datetime.datetime(year, j, 28, 0, 0, 0)
        df = data_train[filter_times(data_train, time1, time2)].copy().reset_index(drop=True)
        for i, (train_index, test_index) in enumerate(ts_split.split(df)):
            model = SARIMAX(endog=df['NP-15 LMP'].loc[train_index], order=non_seas_order, seasonal_order=seas_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(test_size).to_frame(name='predicted')
            eval_df = pd.merge(df[['NP-15 LMP', 'NP-15 LMP Prev_day_price']].loc[test_index].copy(), forecast, left_index=True, right_index=True, how='inner')
            #eval_df = eval_df[eval_df.apply(lambda x: (x['NP-15 LMP'].isna()==False) and (x['NP-15 LMP Prev_day_price'].isna()==False))]
            eval_df = eval_df[(eval_df['NP-15 LMP'].isna()==False)]
            eval_df = eval_df[(eval_df['NP-15 LMP Prev_day_price'].isna()==False)]
            if eval_df.empty == False:
                mse = mean_squared_error(eval_df['NP-15 LMP'], eval_df['predicted'])
                mse_base = mean_squared_error(eval_df['NP-15 LMP'], eval_df['NP-15 LMP Prev_day_price'])
                mse_list.append(mse)
                mse_base_list.append(mse_base)
                print(f"mse for validation in {time2}, fold {i} = {mse}. Baseline = {mse_base}")
            #print(model_fit.summary())
            #mse = mean_squared_error(df['NP-15 LMP'].loc[test_index], model_fit.forecast(test_size))
            #mse_list.append(mse)
            #data_predicted = df.join(pd.concat([df['NP-15 LMP'].loc[train_index].tail(test_size), model_fit.forecast(test_size)]).to_frame(name='predicted'), how='inner')
            #plt.figure(figsize=(18, 4))
            #plt.plot(data_predicted['begin_time'], data_predicted['NP-15 LMP Prev_day_price'] , label='Previous day price')
            #plt.plot(data_predicted['begin_time'], data_predicted['NP-15 LMP'] , label='Actual price')
            #plt.plot(data_predicted['begin_time'], data_predicted['predicted'] , label='Predicted')
            #plt.xticks(rotation=90)
            #plt.legend()
            #plt.show()
        print(f"Month {j} done.")
    
    print(f"Validation error for {year} = {statistics.fmean(mse_list)}, baseline = {statistics.fmean(mse_base_list)}")

In [13]:
crossValidateFinal(2022, 3, 24, (0,1,0, 24), (0,0,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 24), (0,1,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 24), (1,0,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 24), (1,1,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 24), (1,1,1))

#crossValidate(2023, 3, 24, (1,1,0, 24), (0,0,0))
crossValidateFinal(2022, 3, 24, (1,1,0, 24), (0,1,0))
crossValidateFinal(2022, 3, 24, (1,1,0, 24), (1,0,0))
crossValidateFinal(2022, 3, 24, (1,1,0, 24), (1,1,0))

#crossValidate(2023, 3, 24, (1,1,1, 24), (0,0,0))
crossValidateFinal(2022, 3, 24, (1,1,1, 24), (0,1,0))
crossValidateFinal(2022, 3, 24, (1,1,1, 24), (1,0,0))
crossValidateFinal(2022, 3, 24, (1,1,1, 24), (1,1,0))
#crossValidate(2023, 3, 24, (1,1,1, 24), (1,0,1))

Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 0, 0)
Validation error for 2022 = 1389.0700182938288, baseline = 1389.0700182938288
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 1, 0)
Validation error for 2022 = 1524.1914178611112, baseline = 1389.0700182938288
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 0, 0)
Validation error for 2022 = 1365.8851309400459, baseline = 1389.0700182938288
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 1, 0)
Validation error for 2022 = 1746.5596964526694, baseline = 1389.0700182938288
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 1, 1)
Validation error for 2022 = 1376.1632385908051, baseline = 1389.0700182938288
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (0, 1, 0)
Validation error for 2022 = 2638.2455110685873, baseline = 1389.0700182938288
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (1, 0, 0)
Validation error for 2022 = 2783.0239861915034, baseline = 1389.0700182938288
Seasonal orde

KeyboardInterrupt: 

In [ ]:
crossValidateFinal(2022, 3, 24, (0,1,0, 96), (0,0,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 96), (0,1,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 96), (1,0,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 96), (1,1,0))
crossValidateFinal(2022, 3, 24, (0,1,0, 96), (1,1,1))


#crossValidate(2023, 3, 24, (1,1,1, 24), (0,0,0))
crossValidateFinal(2022, 3, 24, (1,1,1, 96), (0,1,0))
crossValidateFinal(2022, 3, 24, (1,1,1, 96), (1,0,0))
crossValidateFinal(2022, 3, 24, (1,1,1, 96), (1,1,0))
#crossValidate(2023, 3, 24, (1,1,1, 24), (1,0,1))

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (0, 0, 0)
5209
mse for validation in 2022-03-28 00:00:00, fold 0 = 313.2912817170417. Baseline = 313.2912817170417
5233
mse for validation in 2022-03-28 00:00:00, fold 1 = 91.75999622250004. Baseline = 91.75999622249999
5257
mse for validation in 2022-03-28 00:00:00, fold 2 = 150.5879616921348. Baseline = 150.5879616921348
5497
mse for validation in 2022-06-28 00:00:00, fold 0 = 772.1105729087167. Baseline = 772.1105729087167
5521
mse for validation in 2022-06-28 00:00:00, fold 1 = 380.37703348811255. Baseline = 380.37703348811255
5545
mse for validation in 2022-06-28 00:00:00, fold 2 = 212.35470666185455. Baseline = 212.35470666185455
5497
mse for validation in 2022-09-28 00:00:00, fold 0 = 119.31897825421247. Baseline = 119.31897825421247
5521
mse for validation in 2022-09-28 00:00:00, fold 1 = 303.7383622826125. Baseline = 303.7383622826125
5545
mse for validation in 2022-09-28 00:00:00, fold 2 = 81.327529078725. Baseline = 81.327

/opt/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


mse for validation in 2022-12-28 00:00:00, fold 0 = 2240.1801080407367. Baseline = 6155.07295344818
5425


/opt/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


mse for validation in 2022-12-28 00:00:00, fold 1 = 1535.6949948941872. Baseline = 13631.195195465481
5449


/opt/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


mse for validation in 2022-12-28 00:00:00, fold 2 = 875.1628024922885. Baseline = 9430.14275047342
Validation error for 2022 = 671.2552192500162, baseline = 2636.7731101410827
Seasonal order - (1, 1, 0, 96), Non-seasonal order - (0, 1, 0)
5209


KeyboardInterrupt: 

In [9]:
crossValidateFinal(2022, 3, 8, (0,1,0,96), (0,0,0))

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (0, 0, 0)
mse for validation in 2022-03-28 00:00:00, fold 0 = 136.90473471142857. Baseline = 136.90473471142857
mse for validation in 2022-03-28 00:00:00, fold 1 = 152.86927358453752. Baseline = 152.86927358453752
mse for validation in 2022-03-28 00:00:00, fold 2 = 160.27947340785002. Baseline = 160.27947340785002
Month 3 done.
mse for validation in 2022-06-28 00:00:00, fold 0 = 220.58507247429998. Baseline = 220.58507247429998
mse for validation in 2022-06-28 00:00:00, fold 1 = 346.5852297600502. Baseline = 346.5852297600502
mse for validation in 2022-06-28 00:00:00, fold 2 = 103.45144852576249. Baseline = 103.45144852576249
Month 6 done.
mse for validation in 2022-09-28 00:00:00, fold 0 = 191.37983698357496. Baseline = 191.379836983575
mse for validation in 2022-09-28 00:00:00, fold 1 = 28.42084690795. Baseline = 28.42084690795
mse for validation in 2022-09-28 00:00:00, fold 2 = 24.18190334464998. Baseline = 24.18190334464998
Month